In [18]:
import numpy as np
import matplotlib.pyplot as plt
import random
import math
import copy
from PackingUtils import update_height_map
import json
from tqdm import  tqdm

In [19]:
# def calculate_height_map(container_dim, items):
#     """
#     Calculates the height map for the bin packing problem.

#     Parameters:
#     - container_dim: Tuple (L, W, H) representing the container dimensions.
#     - items: List of dictionaries, where each item contains:
#         - "position": (x, y, z) (bottom-left-back corner of the item).
#         - "dimensions": (dx, dy, dz) (length, width, height of the item after rotation).

#     Returns:
#     - height_map: 2D list representing the height at each (x, y) coordinate.
#     """
#     L, W, H = container_dim
#     height_map = [[0 for _ in range(W)] for _ in range(L)]

#     for item in items:
#         x, y, z = item["position"]
#         dx, dy, dz = item["dimensions"]

#         for x_offset in range(dx):
#             for y_offset in range(dy):
#                 x_index = x + x_offset
#                 y_index = y + y_offset
                
#                 # Ensure indices are within bounds
#                 if 0 <= x_index < L and 0 <= y_index < W:
#                     height_map[x_index][y_index] = max(height_map[x_index][y_index], z + dz)

#     return height_map

# def calculate_height_map(item_base, item_new):
	


In [28]:

def generate_bin_packing_dataset(container_dim, num_items_range):
    """
    Generates a dataset for the bin packing problem.

    Parameters:
    - container_dim: Tuple (L, W, H) representing the container dimensions.
    - num_items_range: Tuple (min_items, max_items) for the range of items to generate.

    Returns:
    - items: List of generated items with their position, dimensions, and rotations.
    - height_maps: List of height maps after each item is added.
    """
    L, W, H = container_dim
    items = [{"position": (0, 0, 0), "dimensions": (L, W, H)}]  # Start with the full container
    num_items = random.randint(*num_items_range)
    # height_maps = []
    dataset_list = []

    while len(items) < num_items:
        # Choose an item randomly with its volume as weight
        # volumes = [item["dimensions"][0] * item["dimensions"][1] * item["dimensions"][2] for item in items]
        item_weights = [max(item["dimensions"]) for item in items]
        chosen_index = random.choices(range(len(items)), weights=item_weights, k=1)[0]
        chosen_item = items.pop(chosen_index)
        
        dx, dy, dz = chosen_item["dimensions"]
        x, y, z = chosen_item["position"]

        # Choose an axis randomly with its length as weight
        axis_lengths = [dx, dy, dz]
        axis_weights  = [dx**2, dy**2, dz**2]
        chosen_axis = random.choices(range(3), weights=axis_weights, k=1)[0]

        # Choose a random position on the chosen axis
        split_point = random.randint(1, axis_lengths[chosen_axis] - 1)
        # print(f"axis = {chosen_axis}, axis_len = {axis_lengths}, split_point: {split_point}, ")

        # Generate new items based on the split
        if chosen_axis == 0:  # Split along the x-axis
            new_item1 = {"position": (x, y, z), "dimensions": (split_point, dy, dz)}
            new_item2 = {"position": (x + split_point, y, z), "dimensions": (dx - split_point, dy, dz)}
        elif chosen_axis == 1:  # Split along the y-axis
            new_item1 = {"position": (x, y, z), "dimensions": (dx, split_point, dz)}
            new_item2 = {"position": (x, y + split_point, z), "dimensions": (dx, dy - split_point, dz)}
        else:  # Split along the z-axis
            new_item1 = {"position": (x, y, z), "dimensions": (dx, dy, split_point)}
            new_item2 = {"position": (x, y, z + split_point), "dimensions": (dx, dy, dz - split_point)}

        # Randomly rotate the new items
        # def rotate_item(item):
        #     dims = list(item["dimensions"])
        #     random.shuffle(dims)
        #     item["dimensions"] = tuple(dims)

        # rotate_item(new_item1)
        # rotate_item(new_item2)
        # Add the new items to the list
        items.extend([new_item1, copy.deepcopy(new_item2)])
        
        rotate_item_2 = 0
        if random.random() < 0.5:
            new_item2["dimensions"] = (new_item2["dimensions"][1], new_item2["dimensions"][0], new_item2["dimensions"][2])
            rotate_item_2 = 1


        # Calculate the height map
        # Set item1 as base, and item2 is newly added
        # print(f"item 1: {new_item1}")
        # print(f"item 2: {new_item2}, rotate = {rotate_item_2}")
        height_map = update_height_map(
            np.zeros((L, W)), 
            item_size=new_item1["dimensions"],
            item_position=[new_item1["position"][0] + new_item1['dimensions'][0] - 1, new_item1["position"][1]],
            item_orientation=0
        )
        # height_maps.append(height_map)
        dataset_list.append({
            "height_map": height_map.tolist(),
            "new_item": new_item2["dimensions"],
            "position": [new_item2["position"][0] + new_item2["dimensions"][rotate_item_2] - 1, new_item2["position"][1]],
            "rotate": rotate_item_2,
		})

    return items, dataset_list


In [32]:

# Example usage
container_dim = (100, 100, 100)  # Container dimensions
num_items_range = (10, 50)  # Range for the number of items
items, dataset_list = generate_bin_packing_dataset(container_dim, num_items_range)

# Print some results
print(f"Generated {len(items)} items.")
# for i, height_map in enumerate(dataset_list[14:19]):  # Show the first 3 height maps
#     print(f"Height map after item {i + 1}:")
#     for row in height_map:
#         print(height_map[row])

item_list = []
for item in items:
    item_list.append(item["dimensions"])
item_list = np.array(item_list)
print(item_list)
np.save("dataset_map/item_list.npy", item_list)

Generated 50 items.
[[27 41 29]
 [27 17 29]
 [12 11 20]
 [ 9 12 29]
 [18  3 29]
 [32  2 10]
 [32 11 29]
 [32 24 29]
 [ 8  1 29]
 [ 6 47 29]
 [ 4 11  9]
 [ 8 11  9]
 [26 33 29]
 [26 30 13]
 [28 96 71]
 [29 42  3]
 [29 37 26]
 [29  5 26]
 [13  3  8]
 [ 8 28 29]
 [ 8 13 29]
 [ 2  2 19]
 [30  2 19]
 [12 23 27]
 [ 3 27 29]
 [12  9 29]
 [12 15 29]
 [26 17 16]
 [ 6 16 17]
 [ 3 16 12]
 [ 3 16 12]
 [13  3  2]
 [13  3 19]
 [18  4 47]
 [18  4 24]
 [28 27 22]
 [28 27  7]
 [39  4 71]
 [43  4 71]
 [29 11 29]
 [ 1 12 29]
 [21 12 29]
 [24 96 71]
 [48 96 71]
 [12 13  2]
 [12 10  2]
 [26  4 16]
 [26  9 16]
 [12 47 29]
 [17 47 29]]


In [22]:
dataset_size = 50

bar = tqdm(range(dataset_size), desc="Generating datasets")
for t in bar:
	_ , dataset_list = generate_bin_packing_dataset(container_dim, num_items_range)
	with open(f"dataset_map/dataset_{t}.json", "w") as f:
		json.dump(dataset_list, f, indent=4)

Generating datasets:  20%|██        | 10/50 [00:03<00:12,  3.27it/s]


KeyboardInterrupt: 